# Step 1 — Per-slide spatial domain detection with SpaGCN

This notebook runs SpaGCN on the pseudo-spot expression table of a single TCGA slide to identify spatially coherent expression domains. Across TCGA-BRCA, the paper reports an average of ~6.6 domains per slide.

It is adapted from the per-slide logic inside `spatial_clusters/SpaGCN_spatial_clusters_for_ST_pred_slurm.py`, which was originally invoked by a swarm wrapper to loop over all slides. Here we strip the loop and hard-code one slide so the code reads as a tutorial.

**Inputs (replace paths with your own)**
- Pseudo-spot expression CSV: one row per pseudo-spot, columns `x`, `y` (grid coords), `x_px`, `y_px` (pixel coords), plus one column per gene. Values are log10(1+counts).
- Gene list (`overlap_genes.csv`): the gene set used in Path2Space regression — restricts SpaGCN to the predictable-genes subset.
- Slide image (`.tif`) for histology-aware adjacency. Optional in this implementation (`histology=False` below).

**Output**
- `{slide_name}.h5ad` AnnData with `obs['spatial_cluster']` (raw SpaGCN domain) and `obs['refined_pred']` (refined, slide-name-suffixed).

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata
import SpaGCN as spg
from PIL import Image
from skimage.transform import rescale

Image.MAX_IMAGE_PIXELS = None  # disable the decompression-bomb guard for large WSIs

## Configuration

Hard-code one slide and the target number of domains. `target_num=7` is the value used in the paper for the per-slide SpaGCN runs (the actual count is then refined by `search_res`).

In [ ]:
# Example: replace with any TCGA-BRCA slide ID
slide_name = "TCGA-A1-A0SB-01Z-00-DX1"

target_num = 7        # initial target number of SpaGCN domains per slide
pred = 'labels'       # subdir of the pseudo-spot expression source

# === Replace these paths with your own ===
best_pre = '/vf/users/Ruppin_ST/ms/data/overlap_genes.csv'
svs_dir  = '/vf/users/Ruppin_ST/Datasets/TNBC_BC_data/slides/'
exp_input_dir = f'/vf/users/Ruppin_ST/p2s_v2/data/{pred}_new_c/'
dir_out = f'/vf/users/Ruppin_ST/p2s_v2/tnbc_spaGCN_res/{target_num}_clust/{pred}/'
os.makedirs(dir_out, exist_ok=True)

## Helpers

In [ ]:
def create_anndata_from_df(df):
    """Build an AnnData from a pseudo-spot expression table with x, y grid columns."""
    spatial_coords = df[['x', 'y']].values
    gene_expr = df.drop(columns=['x', 'y'])
    adata = anndata.AnnData(
        X=gene_expr.values,
        obs=pd.DataFrame(index=gene_expr.index),
        var=pd.DataFrame(index=gene_expr.columns),
    )
    adata.obsm['spatial'] = spatial_coords
    return adata


def load_image_with_downsampling(tif_file_path, downscale_factor=32):
    """Load a WSI and downsample with skimage. Only used when histology=True in the adjacency."""
    image = Image.open(tif_file_path).convert('RGB')
    image_np = np.array(image)
    return rescale(
        image_np,
        scale=(1 / downscale_factor, 1 / downscale_factor, 1),
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.uint8)

## Load the gene panel and the pseudo-spot expression

The expression table is log10(1+counts); SpaGCN expects linear-scale counts, so we reverse the transform on the modeled genes.

In [ ]:
genes = pd.read_csv(best_pre).iloc[:, 1].tolist()

exp_path = f'{exp_input_dir}{slide_name}.csv'
exp = pd.read_csv(exp_path, index_col=0)

# log10(1+x) -> x for the modeled gene columns
exp.loc[:, genes] = 10 ** exp.loc[:, genes] - 1
exp_p = exp[['x', 'y'] + genes]

slide_data = create_anndata_from_df(exp_p)
slide_data.obs['x_pixel'] = exp.loc[slide_data.obs_names, 'x_px']
slide_data.obs['y_pixel'] = exp.loc[slide_data.obs_names, 'y_px']

spg.prefilter_genes(slide_data, min_cells=3)
spg.prefilter_specialgenes(slide_data)
slide_data

## Adjacency matrix

Build the spot adjacency matrix used by SpaGCN. We pass pixel coordinates but `histology=False` — the paper's per-slide runs use expression+coords only, not the histology channel.

In [ ]:
grid_coords = slide_data.obsm['spatial']
x_coords = grid_coords[:, 0]
y_coords = grid_coords[:, 1]

# Note: the original script swaps these (x_pixel <- y_pixel and vice versa). Preserved as-is.
y_pixel = slide_data.obs['x_pixel'].astype(int)
x_pixel = slide_data.obs['y_pixel'].astype(int)

# If you want a histology-aware adjacency, load the WSI and pass image=image, histology=True.
# svs_file_path = os.path.join(svs_dir, f'{slide_name}.tif')
# image = load_image_with_downsampling(svs_file_path, downscale_factor=32)

x_pixel_scaled = (x_pixel / 32).astype(int)
y_pixel_scaled = (y_pixel / 32).astype(int)

adj = spg.calculate_adj_matrix(
    x_coords, y_coords,
    x_pixel=x_pixel_scaled.values,
    y_pixel=y_pixel_scaled.values,
    image=None,
    histology=False,
)

## Search SpaGCN hyperparameters and train

1. `search_l` chooses the bandwidth `l` so that the average neighborhood probability over all spots is ~0.5.
2. `search_res` chooses the Louvain resolution so SpaGCN returns approximately `target_num` clusters.
3. `SpaGCN().train` runs the model; `predict` returns per-spot domain labels.

In [ ]:
l = spg.search_l(p=0.5, adj=adj, start=0.0001, end=1000, tol=0.01, max_run=100)

res = spg.search_res(
    slide_data, adj, l,
    start=0.7, target_num=target_num, step=0.1, tol=5e-3,
    lr=0.05, max_epochs=20,
    r_seed=100, t_seed=100, n_seed=100,
)

clf = spg.SpaGCN()
clf.set_l(l)
clf.train(slide_data, adj, init_spa=True, init='louvain', res=res,
          tol=5e-3, lr=0.05, max_epochs=200)
y_pred, _ = clf.predict()

## Refine and persist

`spatial_domains_refinement_ez_mode` smooths the per-spot labels using the hexagonal Visium-style neighborhood. The refined labels are suffixed with the slide name so they remain unique when slides are concatenated for cross-patient clustering (step 2).

In [ ]:
slide_data.obs['spatial_cluster'] = y_pred
slide_data.obs['spatial_cluster'] = slide_data.obs['spatial_cluster'].astype('category')

slide_data.obs['refined_pred'] = spg.spatial_domains_refinement_ez_mode(
    sample_id=slide_data.obs.index.tolist(),
    pred=slide_data.obs['spatial_cluster'].tolist(),
    x_array=x_coords, y_array=y_coords,
    shape='hexagon',
)
slide_data.obs['refined_pred'] = slide_data.obs['refined_pred'].astype('category')
slide_data.obs['refined_pred'] = [f'{x}_#_{slide_name}' for x in slide_data.obs['refined_pred'].tolist()]

file_out = f'{dir_out}{slide_name}.h5ad'
slide_data.write(file_out)
print('Wrote:', file_out)

## (Optional) Visualize the domains

Spatial scatter of pseudo-spots colored by `refined_pred`.

In [ ]:
import scanpy as sc
sc.pl.embedding(slide_data, basis='spatial', color='refined_pred')